# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


In [31]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display


In [34]:
load_dotenv(override=True)


google_api_key = os.getenv('GOOGLE_API_KEY')

groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')


if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")



Google API Key exists and begins AI
Groq API Key exists and begins gsk_
OpenRouter API Key exists and begins sk-or-


In [35]:
# Connect to client libraries

openai = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
groq_url = "https://api.groq.com/openai/v1"
openrouter_url = "https://openrouter.ai/api/v1"
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)



In [36]:
models = ["gemini-2.5-pro", "qwen/qwen3-coder-30b-a3b-instruct", "openai/gpt-oss-120b", ]

clients = { "gemini-2.5-pro": gemini, "openai/gpt-oss-120b": groq,"qwen/qwen3-coder-30b-a3b-instruct": openrouter}

# Want to keep costs ultra-low? Replace this with models of your choice, using the examples from yesterday

In [37]:
from system_info import retrieve_system_info, rust_toolchain_info

system_info = retrieve_system_info()
rust_info = rust_toolchain_info()
rust_info

{'installed': False,
 'rustc': {'path': '',
  'version': '',
  'host_triple': '',
  'release': '',
  'commit_hash': ''},
 'cargo': {'path': '', 'version': ''},
 'rustup': {'path': '',
  'version': '',
  'active_toolchain': '',
  'default_toolchain': '',
  'toolchains': [],
  'targets_installed': []},
 'rust_analyzer': {'path': ''},
 'env': {'CARGO_HOME': '',
  'RUSTUP_HOME': '',
  'RUSTFLAGS': '',
  'CARGO_BUILD_TARGET': ''},
 'execution_examples': []}

In [19]:
message = f"""
Here is a report of the system information for my computer.
I want to run a Rust compiler to compile a single rust file called main.rs and then execute it in the simplest way possible.
Please reply with whether I need to install a Rust toolchain to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile Rust code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.
Have the maximum possible runtime performance in mind; compile time can be slow. Fastest possible runtime performance for this platform is key.
Reply with the commands in markdown.

System information:
{system_info}

Rust toolchain information:
{rust_info}
"""

response = openrouter.chat.completions.create(model=models[2], messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))

## 1️⃣ Do you need to install a Rust tool‑chain?  

**Yes.**  
Your system‑information dump shows no `rustc`, `cargo` or `rustup` present:

```json
{
  "installed": false,
  "rustc": { "path": "", "version": "" },
  "cargo": { "path": "", "version": "" },
  "rustup": { "path": "", "version": "" }
}
```

So you must bring a Rust tool‑chain onto the machine before you can compile `main.rs`.

---

## 2️⃣ Fast‑to‑install, “official” Rust tool‑chain on Windows (AMD64)

The easiest way on your Windows 11 machine (you already have `winget`) is to let **rustup** do the heavy lifting. `rustup` will:

* download the correct `rustc` and `cargo` binaries,
* let you pick the *GNU* or *MSVC* ABI (the GNU ABI will work nicely with the MinGW‑w64 compilers you already have),
* keep the tool‑chain up‑to‑date automatically.

### Step‑by‑step installation

| Step | Command (run in an **elevated PowerShell** or **cmd.exe**) | What it does |
|------|-----------------------------------------------------------|--------------|
| 1️⃣ | ```powershell<br>winget install --exact --id Rustlang.Rustup<br>``` | Installs the `rustup‑init.exe` bootstrap program. |
| 2️⃣ | ```powershell<br>rustup-init.exe -y --default-toolchain stable‑x86_64‑pc‑windows‑gnu<br>``` | Runs the installer non‑interactively (`-y`). <br>Installs the **stable** tool‑chain targeting the GNU ABI (`x86_64-pc-windows-gnu`). |
| 3️⃣ | ```powershell<br>refreshenv   # (or just close & reopen the terminal) <br>``` | Makes the newly‑added `~\.cargo\bin` directory visible to the current shell. |
| 4️⃣ | ```powershell<br>rustc --version<br>cargo --version<br>``` | Verify that the tools are reachable. You should see something like `rustc 1.xx.x (yyyy‑mm‑dd)` and `cargo 1.xx.x`. |

> **Alternative (single‑line) via winget**  
> Modern versions of `winget` can install rustup **and** select the GNU ABI in one go:  
> ```powershell
> winget install --exact --id Rustlang.Rustup -e --override "--default-toolchain stable-x86_64-pc-windows-gnu -y"
> ```
> After the install, just run `refreshenv` (or restart the terminal).

---

## 3️⃣ Compile **as fast as possible at runtime** (release‑grade)

You asked for the **maximum possible runtime performance** – compilation time can be long.  
Instead of using `cargo`, you can invoke `rustc` directly on a single file. The following flags give you the fastest generated code for the current CPU:

| Flag | Meaning |
|------|---------|
| `-C opt-level=3` | Highest optimisation level. |
| `-C target-cpu=native` | Generate code for the exact CPU you run on (enables AVX2, SSE4.2, etc.). |
| `-C lto=fat` | Whole‑program Link‑Time Optimisation (best performance). |
| `-C codegen-units=1` | Forces a single code‑generation unit → better optimisation across the whole crate. |
| `-C panic=abort` *(optional)* | Removes unwind code, making binaries a bit smaller and faster. |
| `-C embed-bitcode=no` *(optional)* | Reduces binary size; not needed for runtime speed. |

Putting it together:

```bash
rustc \
  -C opt-level=3 \
  -C target-cpu=native \
  -C lto=fat \
  -C codegen-units=1 \
  -C panic=abort \
  main.rs -o main.exe
```

> **Why not `cargo build --release`?**  
> `cargo` adds a lot of convenience (dependency handling, workspace support) but it also runs a wrapper script that sets a few defaults. For a **single file** the direct `rustc` command above is simpler and gives you full control over the optimisation flags.

---

## 4️⃣ Using the commands from Python

Below is a **self‑contained** Python snippet that compiles `main.rs` with the flags above and then runs the produced executable. It uses `subprocess.run` exactly as you described.

```python
import subprocess
import pathlib

# ----- 1️⃣  Build ---------------------------------------------------------
src = pathlib.Path("main.rs")
exe = pathlib.Path("main.exe")               # Windows executable name

compile_command = [
    "rustc",
    "-C", "opt-level=3",
    "-C", "target-cpu=native",
    "-C", "lto=fat",
    "-C", "codegen-units=1",
    "-C", "panic=abort",
    str(src),
    "-o",
    str(exe),
]

# Run the compiler
compile_result = subprocess.run(
    compile_command,
    check=True,
    text=True,
    capture_output=True,
)

print("⚙️  Compilation succeeded")
print("compiler stdout:", compile_result.stdout)
print("compiler stderr:", compile_result.stderr)

# ----- 2️⃣  Execute -------------------------------------------------------
run_command = [str(exe)]          # on Windows you can also use ".\\main.exe"

run_result = subprocess.run(
    run_command,
    check=True,
    text=True,
    capture_output=True,
)

print("\n🚀  Program output")
print(run_result.stdout)
```

### What the two lists contain

| Variable | Value (as Python list) |
|----------|------------------------|
| **`compile_command`** | ```["rustc","-C","opt-level=3","-C","target-cpu=native","-C","lto=fat","-C","codegen-units=1","-C","panic=abort","main.rs","-o","main.exe"]``` |
| **`run_command`** | ```["main.exe"]``` (or `[".\main.exe"]` if you prefer an explicit relative path) |

These commands give you the **fastest possible runtime** for the compiled binary on your specific CPU, at the cost of a longer compile step (which you said is acceptable).

---

## 5️⃣ Quick sanity‑check after installation

```powershell
> rustc -Vv
rustc 1.78.0 (9c9e0b6c5 2024-02-07)  # version may be newer
host: x86_64-pc-windows-gnu
```

If the host triple is `x86_64-pc-windows-gnu` you’re already using the GNU ABI, which works with your MinGW tool‑chain. If you ever need the MSVC ABI you can switch with:

```powershell
rustup toolchain install stable-x86_64-pc-windows-msvc
rustup default stable-x86_64-pc-windows-msvc
```

But for the compile command shown above **no further changes are required**.

---

### TL;DR

1. **Install Rust** (via `winget` + `rustup-init.exe` → stable‑GNU tool‑chain).  
2. **Compile** with `rustc -C opt-level=3 -C target-cpu=native -C lto=fat -C codegen-units=1 -C panic=abort main.rs -o main.exe`.  
3. **Run** the program with `./main.exe`.  

The two Python list literals you need are:

```markdown
compile_command = [
    "rustc",
    "-C", "opt-level=3",
    "-C", "target-cpu=native",
    "-C", "lto=fat",
    "-C", "codegen-units=1",
    "-C", "panic=abort",
    "main.rs",
    "-o", "main.exe",
]

run_command = ["./main.exe"]   # or ["main.exe"] on Windows
```

That’s it – you’re now ready to compile and run Rust code with maximum runtime performance on your Windows 11 AMD64 machine. Happy coding! 🚀

In [38]:
compile_command = [
    "/Users/ed/.cargo/bin/rustc",
    "main.rs",
    "-C", "opt-level=3",
    "-C", "target-cpu=native",
    "-C", "codegen-units=1",
    "-C", "lto=fat",
    "-C", "panic=abort",
    "-C", "strip=symbols",
    "-o", "main",
]

run_command = ["./main"]


## And now, on with the main task

In [39]:
language = "Rust" # or "C++"
extension = "rs" if language == "Rust" else "cpp"

system_prompt = f"""
Your task is to convert Python code into high performance {language} code.
Respond only with {language} code. Do not provide any explanation other than occasional comments.
The {language} response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to {language} with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.{language} and then compiled and executed; the compilation command is:
{compile_command}
Respond only with {language} code.
Python code to port:

```python
{python}
```
"""

In [40]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [41]:
def write_output(code):
    with open(f"main.{extension}", "w") as f:
        f.write(code)

In [42]:
def port(model, python):
    client = clients[model]
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```rust','').replace('```','')
    return reply

In [43]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [44]:
# Use the commands from GPT 5

def compile_and_run(code):
    write_output(code)
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return run_result.stdout
    except subprocess.CalledProcessError as e:
        return f"An error occurred:\n{e.stderr}"

In [45]:
python_hard = """# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

## RESULTS!

Qwen 2.5 Coder: FAIL  
Gemini 2.5 Pro: FAIL  
DeepSeek Coder v2: FAIL  
Qwen3 Coder 30B: FAIL  
Claude Sonnet 4.5: FAIL    
GPT-5: FAIL    

3rd place: GPT-oss-20B: 0.000341  
2nd place: Grok 4: 0.000317  
**1st place: OpenAI GPT-OSS 120B: 0.000304**  

In [ ]:
print(f"In Ed's experimenet, the GPT-OSS 120B model outcome is {33.755209/0.000304:,.0f} times faster than the Python code.")